# Native CLM v0 M3R — Read-Preserving / Lineage-Isolated Growth

Canonical two-GPU formal workflow. GPU0 runs the frozen M3 global-pool growth control; GPU1 runs the lineage-isolated treatment. Formal seeds 73611/73612/73613 remain untouched until the formal runner cell.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BRANCH = 'codex/native-clm-v0-m3r-read-preserving-growth'
REPO = Path('/kaggle/working/mini-cells')
CHECKPOINT_DIR = Path('/kaggle/working/native-clm-v0-m1')
CHECKPOINT = CHECKPOINT_DIR / 'final-model.pt'
DATA = Path('/kaggle/working/native-clm-m3r-data')
OUT = REPO / 'artifacts/experiments/native-clm-v0-m3r-read-preserving-growth'

def run(cmd, check=True, env=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check, env=env)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/ArcheLabs/mini-cells.git', REPO])
else:
    os.chdir(REPO)
    run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'])
    run(['git', 'checkout', BRANCH])
    run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'])
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'])
print('HEAD:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import auth_check
import torch

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'Missing HF_TOKEN'
assert os.environ['GITHUB_TOKEN'], 'Missing GITHUB_TOKEN'
assert torch.cuda.is_available(), 'CUDA required for canonical M3R'
assert torch.cuda.device_count() >= 2, f'M3R requires two GPUs, found {torch.cuda.device_count()}'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
auth_check('archelabsxyz/native-clm-v0', repo_type='model', token=os.environ['HF_TOKEN'], write=True)
print('HF model repository write preflight: PASS')

In [ ]:
run([
    sys.executable, 'scripts/research/fetch_native_clm_v0_m1_checkpoint.py',
    '--repo-id', 'archelabsxyz/native-clm-v0',
    '--filename', 'final-model.pt',
    '--expected-sha256', '91cc66f744c97e50105acbb7cdc328a95cb87a32c49baf5b0d6e462d4d4c4c7f',
    '--output', CHECKPOINT,
])
print((CHECKPOINT_DIR / 'provenance.json').read_text())

In [ ]:
# Create a fresh exact Hub-revision-pinned A/B/C/D snapshot shared by both causal arms.
run([sys.executable, 'scripts/research/prepare_native_clm_v0_m3r_data.py', '--output-dir', DATA])
manifest = json.loads((DATA / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2))

In [ ]:
# Refuse a second supposedly untouched formal decision after canonical publication.
tracked_decision = 'artifacts/experiments/native-clm-v0-m3r-read-preserving-growth/decision.json'
tracked = subprocess.run(['git', 'ls-files', '--error-unmatch', tracked_decision], capture_output=True).returncode == 0
assert not tracked, 'Canonical M3R decision is already tracked; later runs are reproduction only.'
formal = run([
    sys.executable, 'scripts/research/run_native_clm_v0_m3r.py',
    '--formal',
    '--checkpoint', CHECKPOINT,
    '--data-dir', DATA,
    '--output-dir', OUT,
    '--devices', '0,1',
], check=False)
assert formal.returncode in (0, 2), f'unexpected formal runner return code {formal.returncode}'
print('formal runner return code:', formal.returncode)

In [ ]:
decision = json.loads((OUT / 'decision.json').read_text())
print(json.dumps({
    'status': decision['status'],
    'scientific_decision': decision['scientific_decision'],
    'protocol_sha256': decision['protocol_sha256'],
    'data_manifest_sha256': decision['data_manifest_sha256'],
    'completed_seeds': decision['completed_seeds'],
}, indent=2))
for seed in decision['seed_results']:
    print('seed', seed['seed'], 'pass=', seed['pass'], 'global_A=', seed['global_A_regression'], 'lineage_A=', seed['lineage_A_regression'], 'advantage=', seed['A_retention_advantage'], 'birth_drift=', seed['max_birth_logits_max_abs_drift'], 'A_child_share_reduction=', seed['A_child_share_reduction_vs_global'])
    for gate, passed in seed['gates'].items():
        print('  ', gate, passed)

In [ ]:
# Upload all six end-state checkpoints first, then Git-publish lightweight evidence.
run([
    sys.executable, 'scripts/research/publish_native_clm_v0_m3r.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
    '--checkpoint-provenance', CHECKPOINT_DIR / 'provenance.json',
    '--data-manifest', DATA / 'manifest.json',
    '--hf-repo', 'archelabsxyz/native-clm-v0',
    '--require-hf-upload',
])
print('Published M3R status:', decision['status'])